## simple_triton example

This notebook illustrates how to use the simple_triton package to perform inference on tiles from a whole-slide image using a Triton inference server.

Notes for running:
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount the EfficientNetV2S.tensorflow savedmodel directory to the triton container
- Load the model (below)

In [ ]:
# install large_image with tile sources
!apt update
!apt install -y python3-openslide openslide-tools
!pip install ../../histomics_stream 'large_image[tiff,openslide]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install ../../simple_triton

# install mil
!pip install ../../mil

## Create a histomics stream study

Parameters in this cell are for reading from the whole-slide image (magnification, tile size, tile overlap, mask file).

In [ ]:
from mil.io.utils import study

# slide parameters
batch = 64
magnification = 20
tile = 224
overlap = 0
chunk = 224
mask_threshold = 0.5
wsi_path = (
    "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.svs"
)
mask_path = "/tf/notebooks/TCGA-AN-A0G0-01Z-00-DX1.BE0BB5DF-DEDA-48D8-B5D8-2735C767F28F.mask.png"

# create a histomics-stream study from a wsi/mask pair
hs_study = study(
    (wsi_path, mask_path), t=(tile, tile), chunk=(tile, tile), target=20, source="exact"
)

## Create and load model

The function `feature_extractor` can be used to create feature extraction models in the model repository. Note - this cell will take time as the model is downloaded, saved, and loaded into triton. 

Here we generate a model, load the model into triton, and verify that the model state is "READY".

Parameters in this stage include the inference server (address), the model (model name, maximum batch size).

In [ ]:
import json
from google.protobuf.json_format import MessageToDict
import numpy as np
from simple_triton.feature_extraction import feature_extractor
from simple_triton.model import model_config
import tritonclient.grpc as grpcclient

# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server
keras_name = "EfficientNetV2S"
model_name = f"{keras_name}.tensorflow"  # set model name

# create the model and capture output dimensionality
try:
    dimension_output = feature_extractor(
        "/tf/notebooks/models", keras_name, model_name, t=(tile, tile), pool="avg"
    )
except:
    print(f"Model {model_name} already exists.")

# create triton client
client = grpcclient.InferenceServerClient(url=url, verbose=True)

# load tensorflow model with larger batch size
config = {"maxBatchSize": 256}
client.load_model(model_name, config=json.dumps(config))

# check readiness
client.get_model_repository_index()

# deleting the client in main prevents conflicts with child process clients
del client

## Run the inference

Parameters here include the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [ ]:
from simple_triton.feature_extraction import histomics_stream_inference
from simple_triton.submitter import analyze
import time

# inference parameters
batch = 64
limit = 10  # limit on number of pending requests per worker
workers = 32  # total number of Submitter workers
verbose = True  # set verbose as False

# start timer
start = time.time()

# inference
features, results, times = histomics_stream_inference(
    hs_study,
    model_name,
    url="localhost:8001",
    batch=batch,
    workers=workers,
    limit=limit,
)

# display elapsed time
print(f"Total elapsed time: {time.time()-start}")

# analyze performance
analyze(times)